# 🚀 High-Performance GPU Acceleration Demo
### AI Retail Optimization Suite — Powered by NVIDIA RAPIDS & RTX 3050

This notebook demonstrates the massive performance leap achieved by moving from CPU-bound calculations to **GPU-accelerated** machine learning using the **RAPIDS** ecosystem (`cuDF` and `cuML`).

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

try:
    import cudf
    import cuml
    from cuml.ensemble import RandomForestClassifier as cuRF
    GPU_AVAILABLE = True
    print("✅ NVIDIA GPU Acceleration (RAPIDS) is ACTIVE")
except ImportError:
    GPU_AVAILABLE = False
    print("❌ RAPIDS not found. Please wait for the background installation to finish.")

In [ ]:
# Load Large Dataset (100k+ rows)
data_path = r"../data/raw/ecommerce_customer_data_custom_ratios.csv"
df = pd.read_csv(data_path)
print(f"📊 Dataset Loaded: {len(df):,} rows")

# Simple Preprocessing
df = df.dropna()
for col in ['Gender', 'Product Category', 'Payment Method']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df[['Customer Age', 'Gender', 'Product Category', 'Product Price', 'Quantity', 'Total Purchase Amount', 'Payment Method']]
y = df['Churn']

## Phase 1: Standard CPU Processing (Scikit-Learn)
Training a Random Forest with 100 trees on 100,000 rows using CPU cores.

In [ ]:
start_time = time.time()
cpu_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
cpu_model.fit(X, y)
cpu_duration = time.time() - start_time

print(f"🐢 CPU Training Time: {cpu_duration:.2f} seconds")

## Phase 2: GPU-Accelerated Processing (NVIDIA RAPIDS)
Training the same model using thousands of CUDA cores on the RTX 3050.

In [ ]:
if GPU_AVAILABLE:
    # Move data to GPU Memory (VRAM)
    X_gpu = cudf.from_pandas(X)
    y_gpu = cudf.from_pandas(y)

    start_time = time.time()
    gpu_model = cuRF(n_estimators=100, n_streams=1)
    gpu_model.fit(X_gpu, y_gpu)
    gpu_duration = time.time() - start_time

    print(f"🚀 GPU Training Time: {gpu_duration:.2f} seconds")
    print(f"📈 Speedup Factor: {cpu_duration / gpu_duration:.1f}x FASTER")
else:
    print("⚠️ GPU Acceleration not available yet. Please complete the setup.")